# Automated Multilingual CEFR Classification — Kaggle Runner

Runs **Exp 0 – Exp 4** (baselines → CORAL → LLaMA+LoRA) on Kaggle GPU (H100 / T4).

| Experiment | Type | GPU? |
|---|---|---|
| Exp 0 – Majority | baseline | — |
| Exp 1 – TF-IDF+LR | baseline | — |
| Exp 7 – TF-IDF+LinearSVC | baseline | — |
| Exp 9 – Word TF-IDF+LR | ablation | — |
| Exp 2 – XLM-R fine-tuned | transformer | ✓ |
| Exp 3 – CORAL ordinal | transformer | ✓ |
| Exp 8 – Zero-shot XLM-R | zero-shot | ✓ |
| Exp 4 – LLaMA-3.2+LoRA (raw + constrained) | LLM | ✓ HF_TOKEN |

> **HF_TOKEN** — add your Hugging Face token as a Kaggle Secret named `HF_TOKEN`

---

### ⚠️ Run order matters
**Cell 1** detects GPU/CUDA — no torch import yet.  
**Cell 2** installs correct PyTorch wheel and **restarts the kernel automatically**.  
**Cell 3** onwards runs after the restart with a clean process.  
Re-run **Cell 3 → end** after the restart (Kaggle resumes automatically when using `do_shutdown`).

## Cell 1 — GPU Detection (no torch import)

In [ ]:
# ── DO NOT import torch here — it would lock the old wheel into memory ────────
# We use only subprocess (nvidia-smi / nvcc) to detect CUDA version.
import subprocess, re, os, sys

# GPU name + VRAM + compute capability (no torch needed)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU     :", smi.stdout.strip() or "none")

# CUDA toolkit version via nvcc
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
m = re.search(r"release (\d+\.\d+)", nvcc.stdout)
CUDA_VER   = m.group(1) if m else "12.1"
CUDA_MAJOR = int(CUDA_VER.split(".")[0])
CUDA_MINOR = int(CUDA_VER.split(".")[1])
print(f"CUDA    : {CUDA_VER}  (from nvcc)")

# Pick the matching PyTorch wheel index
if   CUDA_MAJOR >= 12 and CUDA_MINOR >= 4: WHL = "cu124"
elif CUDA_MAJOR >= 12:                     WHL = "cu121"
else:                                       WHL = "cu118"
print(f"Wheel   : {WHL}  → https://download.pytorch.org/whl/{WHL}")

## Cell 2 — Install + Kernel Restart

Installs the correct PyTorch wheel for this GPU, then **restarts the kernel**.  
Kaggle will automatically continue from Cell 3 after the restart.

In [ ]:
import subprocess, re, os, sys

SENTINEL = "/kaggle/working/.deps_installed"

if os.path.exists(SENTINEL):
    print("✓ Deps already installed — skipping reinstall")
    print("  (delete /kaggle/working/.deps_installed to force re-run)")
else:
    # Re-detect CUDA version (self-contained)
    nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    m = re.search(r"release (\d+\.\d+)", nvcc.stdout)
    CUDA_VER   = m.group(1) if m else "12.1"
    CUDA_MAJOR = int(CUDA_VER.split(".")[0])
    CUDA_MINOR = int(CUDA_VER.split(".")[1])
    if   CUDA_MAJOR >= 12 and CUDA_MINOR >= 4: WHL = "cu124"
    elif CUDA_MAJOR >= 12:                     WHL = "cu121"
    else:                                       WHL = "cu118"
    IDX = f"https://download.pytorch.org/whl/{WHL}"
    print(f"CUDA {CUDA_VER} → installing torch from {IDX}")

    # 1. PyTorch >= 2.4 ensures full sm_90 (H100) kernel support
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--force-reinstall", "--upgrade",
         "torch>=2.4.0", "torchvision", "torchaudio",
         "--index-url", IDX],
        check=True,
    )
    print("torch ✓")

    # 2. bitsandbytes >= 0.44 ships pre-built sm_90 kernels
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--force-reinstall", "bitsandbytes>=0.44.0"],
        check=True,
    )
    print("bitsandbytes ✓")

    # 3. Remaining packages
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "datasets>=2.14.0", "transformers>=4.44.0",
         "accelerate>=0.34.0", "peft>=0.12.0",
         "evaluate>=0.4.0", "scikit-learn>=1.3.0", "langdetect>=1.0.9"],
        check=True,
    )
    print("deps ✓")

    # Write sentinel BEFORE restart so next run skips install
    open(SENTINEL, "w").close()
    print("
Restarting kernel to load new torch wheel...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)


## Cell 3 — Verify GPU (first torch import, post-restart)

Everything below runs in a fresh Python process with the correct torch wheel loaded.

In [ ]:
import torch, os, sys, subprocess

print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    dev = torch.cuda.get_device_name(0)
    cc  = torch.cuda.get_device_capability(0)
    x   = torch.tensor([1.0, 2.0]).cuda()
    print(f"GPU: {dev}  sm_{cc[0]}{cc[1]}  — tensor OK: {x.tolist()}")
    try:
        import bitsandbytes as bnb
        print(f"bitsandbytes {bnb.__version__} ✓")
    except Exception as e:
        print(f"bitsandbytes warning: {e}")
else:
    print("WARNING: no CUDA device — transformer/LLM cells will be very slow")

In [ ]:
REPO_URL = "https://github.com/huynhduc0/itmo-vkr-cefr.git"  # ← update if forked
REPO_DIR = "/kaggle/working/itmo-vkr-cefr"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"Already cloned: {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    import huggingface_hub
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HF_TOKEN loaded")
except Exception as e:
    print(f"⚠️  HF_TOKEN not available ({e}). Exp 4 will be skipped.")

## Configuration

In [ ]:
LANGUAGE = "en"       # en | ru | it | es | de | fr
TASK     = "sentence" # sentence | essay

EXPS_CPU         = [0, 1, 7, 9]
EXPS_TRANSFORMER = [2, 3, 8]
EXPS_LLM         = [4] if HF_TOKEN else []

BATCH_SIZE_TRANSFORMER = 64   # H100 80 GB — default 16
BATCH_SIZE_LLM         = 8
NUM_EPOCHS             = 5
NUM_EPOCHS_LLM         = 3
USE_BF16               = True  # H100/A100 prefer bf16

print(f"Language={LANGUAGE}  Task={TASK}")
print(f"CPU={EXPS_CPU}  GPU={EXPS_TRANSFORMER}  LLM={EXPS_LLM}")

In [ ]:
from src import config as cfg

cfg.TRANSFORMER_CONFIG["batch_size"] = BATCH_SIZE_TRANSFORMER
cfg.TRANSFORMER_CONFIG["num_epochs"] = NUM_EPOCHS
cfg.LLM_CONFIG["batch_size"]         = BATCH_SIZE_LLM
cfg.LLM_CONFIG["num_epochs"]         = NUM_EPOCHS_LLM

if USE_BF16:
    os.environ["ACCELERATE_MIXED_PRECISION"]        = "bf16"
    os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
    cfg.LLM_CONFIG["bnb_4bit_compute_dtype"]         = "bfloat16"
    print("✓ bf16 enabled")

print(f"Transformer: batch={cfg.TRANSFORMER_CONFIG['batch_size']}, epochs={cfg.TRANSFORMER_CONFIG['num_epochs']}")
print(f"LLM        : batch={cfg.LLM_CONFIG['batch_size']}, epochs={cfg.LLM_CONFIG['num_epochs']}")

## Data Preparation

`prepare_data.py` always creates **both** tracks (sentence + essay) — no `--task` flag.

In [ ]:
DATA_DIR = "/kaggle/working/data"

cmd = [sys.executable, "-m", "src.prepare_data", "--language", LANGUAGE, "--output", DATA_DIR]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("prepare_data failed")

for track in ["sentence", "essay"]:
    for split in ["train", "dev", "test"]:
        path = os.path.join(DATA_DIR, track, f"{split}.jsonl")
        if os.path.exists(path):
            n = sum(1 for _ in open(path, encoding="utf-8"))
            print(f"  {track}/{split}: {n:,}")

In [ ]:
from src.run_experiments import _load_splits_from_jsonl
from src.data_utils import set_seed

set_seed(42)
(train_texts, train_labels), (val_texts, val_labels), (test_texts, test_labels) = \
    _load_splits_from_jsonl(DATA_DIR, TASK)

print(f"Track={TASK}  Train={len(train_texts):,}  Val={len(val_texts):,}  Test={len(test_texts):,}")

from collections import Counter
from src.config import ID2LABEL
print("Labels:", dict(sorted(Counter(ID2LABEL[l] for l in train_labels).items())))

## CPU Baselines (Exp 0, 1, 7, 9)

In [ ]:
from src.run_experiments import run_exp0, run_exp1, run_exp7, run_exp9
import time

all_results = []

if 0 in EXPS_CPU:
    r = run_exp0(train_labels, test_labels, len(test_texts), track=TASK)
    all_results.append(r)
    print(f"Exp 0  QWK={r.qwk:.4f}  Acc={r.accuracy:.4f}  [{r.note}]")

if 1 in EXPS_CPU:
    t0 = time.time()
    r = run_exp1(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"Exp 1  QWK={r.qwk:.4f}±{r.qwk_ci:.3f}  F1={r.macro_f1:.4f}  Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

if 7 in EXPS_CPU:
    t0 = time.time()
    r = run_exp7(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"Exp 7  QWK={r.qwk:.4f}  F1={r.macro_f1:.4f}  Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

if 9 in EXPS_CPU:
    t0 = time.time()
    r = run_exp9(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"Exp 9  QWK={r.qwk:.4f}  F1={r.macro_f1:.4f}  Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

## Transformer Experiments (Exp 2, 3, 8) — GPU

In [ ]:
import time
r2 = None
if 2 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp2
    print("▶ Exp 2 – XLM-R CE")
    t0 = time.time()
    r2 = run_exp2(
        train_texts, train_labels, val_texts, val_labels, test_texts, test_labels,
        track=TASK, language=LANGUAGE,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE_TRANSFORMER, seed=42,
    )
    print(f"   QWK={r2.qwk:.4f}±{r2.qwk_ci:.3f}  F1={r2.macro_f1:.4f}  Acc={r2.accuracy:.4f}  ({(time.time()-t0)/60:.1f} min)")
    all_results.append(r2)

In [ ]:
import time
r3 = None
if 3 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp3
    print("▶ Exp 3 – CORAL (fixed threshold init)")
    t0 = time.time()
    r3 = run_exp3(
        train_texts, train_labels, val_texts, val_labels, test_texts, test_labels,
        track=TASK, language=LANGUAGE,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE_TRANSFORMER, seed=42,
    )
    print(f"   QWK={r3.qwk:.4f}±{r3.qwk_ci:.3f}  F1={r3.macro_f1:.4f}  Acc={r3.accuracy:.4f}  ({(time.time()-t0)/60:.1f} min)")
    if r2: print(f"   CORAL vs CE: ΔQWK={r3.qwk - r2.qwk:+.4f}")
    all_results.append(r3)

In [ ]:
import time
r8 = None
if 8 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp8
    print(f"▶ Exp 8 – Zero-shot XLM-R (en→{LANGUAGE})")
    t0 = time.time()
    r8 = run_exp8(
        train_texts=train_texts, train_labels=train_labels,
        test_texts=test_texts,   test_labels=test_labels,
        track=TASK, language=LANGUAGE, mode="zero_shot", seed=42,
    )
    print(f"   QWK={r8.qwk:.4f}±{r8.qwk_ci:.3f}  F1={r8.macro_f1:.4f}  ({(time.time()-t0)/60:.1f} min)")
    if r2: print(f"   Transfer cost: ΔQWK={r8.qwk - r2.qwk:+.4f}")
    all_results.append(r8)

## LLM Experiment (Exp 4) — LLaMA-3.2 + LoRA

Requires `HF_TOKEN` + LLaMA-3.2 license accepted on HF Hub.  
Returns **raw** (regex, hallucination baseline) + **constrained** (log-prob scoring).

In [ ]:
import time
if 4 in EXPS_LLM:
    from src.run_experiments import run_exp4
    print("▶ Exp 4 – LLaMA-3.2-3B + QLoRA 4-bit")
    t0 = time.time()
    r4_raw, r4_constrained = run_exp4(
        train_texts, train_labels, val_texts, val_labels, test_texts, test_labels,
        track=TASK, language=LANGUAGE, seed=42,
    )
    elapsed = (time.time() - t0) / 60
    print(f"   RAW         QWK={r4_raw.qwk:.4f}  F1={r4_raw.macro_f1:.4f}  [{r4_raw.note}]")
    print(f"   CONSTRAINED QWK={r4_constrained.qwk:.4f}±{r4_constrained.qwk_ci:.3f}  F1={r4_constrained.macro_f1:.4f}  ({elapsed:.1f} min)")
    all_results.extend([r4_raw, r4_constrained])
else:
    print("Exp 4 skipped")

## Results

In [ ]:
from src.run_experiments import print_comparison_table
print(f"Results — {LANGUAGE} / {TASK}")
print_comparison_table(all_results)

In [ ]:
import json
from src.run_experiments import save_results_to_files

OUT_DIR = f"/kaggle/working/results/{TASK}/{LANGUAGE}"
save_results_to_files(all_results, OUT_DIR)

with open(os.path.join(OUT_DIR, "results.json")) as f:
    records = json.load(f)

print(f"\n{'Experiment':<48} {'QWK':>10} {'±CI':>7} {'F1':>8} {'Acc':>7}")
print("-" * 84)
for rec in records:
    ci = f"±{rec['qwk_ci']:.3f}" if rec.get("qwk_ci") else ""
    print(f"{rec['name']:<48} {rec['qwk']:>10.4f} {ci:>7} {rec['macro_f1']:>8.4f} {rec['accuracy']:>7.4f}")

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, numpy as np

names  = [r["name"].replace(" – ", "\n") for r in records]
qwks   = [r["qwk"] for r in records]
cis    = [r.get("qwk_ci", 0.0) for r in records]

def _color(n):
    if "LLM" in n or "LoRA" in n:          return "#e07b39"
    if "CORAL" in n or "Ordinal" in n:      return "#5b9bd5"
    if "Transformer" in n or "XLM-R" in n: return "#4caf50"
    return "#9e9e9e"

fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.6)))
y = np.arange(len(names))
ax.barh(y, qwks, xerr=cis, align="center", height=0.6,
        color=[_color(r["name"]) for r in records],
        capsize=4, error_kw={"elinewidth": 1.5})
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("QWK"); ax.set_xlim(0, 1.05); ax.invert_yaxis()
ax.set_title(f"CEFR {LANGUAGE.upper()}/{TASK} — QWK ± 95% CI", fontsize=12)
for i, (v, ci) in enumerate(zip(qwks, cis)):
    ax.text(min(v + 0.01, 1.0), i,
            f"{v:.3f}" + (f"±{ci:.3f}" if ci else ""), va="center", fontsize=8)
ax.legend(handles=[
    mpatches.Patch(color="#9e9e9e", label="Baseline"),
    mpatches.Patch(color="#4caf50", label="Transformer"),
    mpatches.Patch(color="#5b9bd5", label="CORAL"),
    mpatches.Patch(color="#e07b39", label="LLM"),
], loc="lower right", fontsize=9)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "qwk_bar.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show(); print(f"Saved → {plot_path}")

## (Optional) Multilingual Sweep — All 6 Languages

In [ ]:
# UNCOMMENT to sweep all languages (no --task flag in prepare_data)
# from src.config import SUPPORTED_LANGUAGES
# from src.run_experiments import (
#     run_exp0, run_exp1, run_exp2, run_exp8,
#     _load_splits_from_jsonl, save_results_to_files, print_comparison_table,
# )
# from src.data_utils import set_seed
#
# ALL_RESULTS = {}
# for lang in SUPPORTED_LANGUAGES:
#     print(f"\n{'='*50}  {lang.upper()}")
#     lang_data = f"/kaggle/working/data_{lang}"
#     subprocess.run(
#         [sys.executable, "-m", "src.prepare_data", "--language", lang, "--output", lang_data],
#         cwd=REPO_DIR, check=True,
#     )
#     (tr_t, tr_l), (vl_t, vl_l), (te_t, te_l) = _load_splits_from_jsonl(lang_data, TASK)
#     set_seed(42)
#     lang_results = [
#         run_exp0(tr_l, te_l, len(te_t), track=TASK),
#         run_exp1(tr_t, tr_l, te_t, te_l, track=TASK),
#         run_exp2(tr_t, tr_l, vl_t, vl_l, te_t, te_l, track=TASK, language=lang, seed=42),
#     ]
#     if lang != "en":
#         lang_results.append(
#             run_exp8(tr_t, tr_l, te_t, te_l, track=TASK, language=lang, mode="zero_shot", seed=42)
#         )
#     save_results_to_files(lang_results, f"/kaggle/working/results/{TASK}/{lang}")
#     ALL_RESULTS[lang] = lang_results
#     print_comparison_table(lang_results)
# print("\n✓ All languages done")
print("Multilingual sweep — uncomment to run")

In [ ]:
print("=" * 60)
print(f"Language : {LANGUAGE}  |  Task : {TASK}")
print(f"Results  : {OUT_DIR}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if all_results:
    best = max(all_results, key=lambda r: r.qwk)
    print(f"Best QWK : {best.qwk:.4f}±{best.qwk_ci:.3f}  ({best.name})")
print("=" * 60)